In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import desc
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 08:54:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Inspect Label Store 
gold_label_store_directory = "./datamart/gold/label_store/" 

folder_path = gold_label_store_directory
files_list = glob.glob(os.path.join(folder_path, '*'))

df = spark.read.parquet(*files_list)
print("row_count:", df.count())
df.show()

# print("Unique snapshot dates (Latest first):")
df.select("snapshot_date").distinct().orderBy(desc("snapshot_date")).show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

[Stage 5:==============>                                           (3 + 9) / 12]

+-------------+
|snapshot_date|
+-------------+
|   2024-12-01|
|   2024-11-01|
|   2024-10-01|
|   2024-09-01|
|   2024-08-01|
|   2024-07-01|
|   2024-06-01|
|   2024-05-01|
|   2024-04-01|
|   2024-03-01|
|   2024-02-01|
|   2024-01-01|
|   2023-12-01|
|   2023-11-01|
|   2023-10-01|
|   2023-09-01|
|   2023-08-01|
|   2023-07-01|
+-------------+



In [4]:
# set up config
model_train_date_str = "2024-12-01"
train_test_period_months = 12
oot_period_months = 5
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 12, 1, 0, 0),
 'model_train_date_str': '2024-12-01',
 'oot_end_date': datetime.datetime(2024, 11, 30, 0, 0),
 'oot_period_months': 5,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)}


In [5]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [6]:
# From Lab 5
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

[Stage 13:>                                                       (0 + 12) / 12]

extracted labels_sdf 8476 2023-07-01 00:00:00 2024-11-30 00:00:00


In [7]:
# From Lab 5
# get features
feature_location = "data/feature_clickstream.csv"

# Load CSV into DataFrame - connect to feature store
features_store_sdf = spark.read.csv(feature_location, header=True, inferSchema=True)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()

row_count: 215376
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|  63| 118|  80| 121|  55| 193| 111| 112|-101|   83|  164|  105|  -16|  -81| -126|  114|   35|   85|  -73|   76| CUS_0x1037|   2023-01-01|
|-108| 182| 123|   4| -56|  27|  25|  -6| 284|  222|  203|  190|  -14|  -96|  200|   35|  130|   94|  111|   75| CUS_0x1069|   2023-01-01|
| -13|   8|  87| 166| 214| -98| 215| 152| 129|  139|   14|  203|   26|   86|  171|  125| -130|  354|   17|  302| CUS_0x114a|   2023-01-01|
| -85|  45| 200|  89| 128|  54|  76|  51|  61|  139|    6|  197|  172|   96|  174|  163|   37|  207|  180|  118| CUS_0x1184|   2023-01-01|
|  55| 12

In [8]:
feat_filter_start = config["train_test_start_date"] - relativedelta(months=6)
feat_filter_end = config["oot_end_date"] - relativedelta(months=6)

features_sdf = features_store_sdf.filter(
    (col("snapshot_date") >= feat_filter_start) & 
    (col("snapshot_date") <= feat_filter_end)
)
print(features_sdf.count())


152558


In [9]:
from pyspark.sql.functions import regexp_extract, to_date

labels_with_orig = labels_sdf.withColumn(
    "origination_date",
    to_date(regexp_extract(col("loan_id"), r"(\d{4}_\d{2}_\d{2})$", 1), "yyyy_MM_dd")
)
features_renamed = features_sdf.withColumnRenamed("snapshot_date", "origination_date")

data_pdf = labels_with_orig.join(features_renamed, on=["Customer_ID", "origination_date"], how="left").toPandas()
print(data_pdf.shape)
print(data_pdf.isnull().sum())


[Stage 25:==============>                                          (3 + 9) / 12]

(8476, 26)
Customer_ID         0
origination_date    0
loan_id             0
label               0
label_def           0
snapshot_date       0
fe_1                0
fe_2                0
fe_3                0
fe_4                0
fe_5                0
fe_6                0
fe_7                0
fe_8                0
fe_9                0
fe_10               0
fe_11               0
fe_12               0
fe_13               0
fe_14               0
fe_15               0
fe_16               0
fe_17               0
fe_18               0
fe_19               0
fe_20               0
dtype: int64


In [10]:
feat_filter_start = config["train_test_start_date"] - relativedelta(months=6)
feat_filter_end = config["oot_end_date"] - relativedelta(months=6)

features_sdf = features_store_sdf.filter(
    (col("snapshot_date") >= feat_filter_start) & 
    (col("snapshot_date") <= feat_filter_end)
)
print("extracted features_sdf", features_sdf.count(), feat_filter_start, feat_filter_end)


extracted features_sdf 152558 2023-01-01 00:00:00 2024-05-30 00:00:00


In [11]:
from pyspark.sql.functions import regexp_extract, to_date

labels_with_orig = labels_sdf.withColumn(
    "origination_date",
    to_date(regexp_extract(col("loan_id"), r"(\d{4}_\d{2}_\d{2})$", 1), "yyyy_MM_dd")
)
features_renamed = features_sdf.withColumnRenamed("snapshot_date", "origination_date")

data_pdf = labels_with_orig.join(features_renamed, on=["Customer_ID", "origination_date"], how="left").toPandas()
data_pdf


,Customer_ID,origination_date,loan_id,label,label_def,snapshot_date,fe_1,fe_2,fe_3,fe_4,...,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
0,CUS_0x1136,2023-06-01,CUS_0x1136_2023_06_01,1,30dpd_6mob,2023-12-01,10,48,36,402,...,81,231,206,-36,207,243,-2,281,258,73
1,CUS_0x113e,2023-02-01,CUS_0x113e_2023_02_01,1,30dpd_6mob,2023-08-01,51,286,109,64,...,-6,9,140,-50,161,118,137,148,42,262
2,CUS_0x117d,2023-09-01,CUS_0x117d_2023_09_01,0,30dpd_6mob,2024-03-01,52,-20,230,-143,...,75,112,-85,85,74,173,43,75,-12,100
3,CUS_0x117f,2023-11-01,CUS_0x117f_2023_11_01,0,30dpd_6mob,2024-05-01,145,348,68,230,...,368,132,147,49,146,215,29,53,236,62
4,CUS_0x122f,2024-05-01,CUS_0x122f_2024_05_01,1,30dpd_6mob,2024-11-01,-33,69,121,207,...,211,274,310,-36,123,146,-13,232,12,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8471,CUS_0xf45,2023-05-01,CUS_0xf45_2023_05_01,1,30dpd_6mob,2023-11-01,99,135,51,221,...,221,343,178,70,54,-128,75,100,175,81
8472,CUS_0xf4d,2023-03-01,CUS_0xf4d_2023_03_01,0,30dpd_6mob,2023-09-01,198,52,54,28,...,148,207,33,163,141,95,119,185,46,28
8473,CUS_0xfa7,2023-03-01,CUS_0xfa7_2023_03_01,0,30dpd_6mob,2023-09-01,213,146,-72,-34,...,21,157,89,269,113,19,202,163,145,1
8474,CUS_0xfb4,2023-12-01,CUS_0xfb4_2023_12_01,0,30dpd_6mob,2024-06-01,142,-5,50,49,...,82,10,58,-64,144,322,50,218,232,-66


In [12]:
# From Lab5
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())]

feature_cols = [fe_col for fe_col in data_pdf.columns if fe_col.startswith('fe_')]

X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], train_test_pdf["label"], 
    test_size= 1 - config["train_test_ratio"],
    random_state=88,     # Ensures reproducibility
    shuffle=True,        # Shuffle the data before splitting
    stratify=train_test_pdf["label"]           # Stratify based on the label column
)

print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

X_train 4766
X_test 1192
X_oot 2518
y_train 4766 0.28
y_test 1192 0.28
y_oot 2518 0.3


,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
215,230,242,272,176,255,31,279,6,80,105,225,155,91,32,123,184,195,20,44,161
5213,181,-31,76,-33,117,133,5,208,89,-10,37,92,-26,205,115,153,71,245,85,120
4343,134,-117,142,37,143,125,237,-88,200,163,253,78,-12,52,-122,113,-26,9,36,100
601,102,110,152,104,172,108,269,42,-64,81,-8,15,238,192,200,129,127,92,-39,-61
5268,75,207,-30,92,74,106,75,-209,37,-11,73,21,-173,-92,-33,157,44,128,223,155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426,273,170,194,199,-132,33,246,119,176,54,253,70,175,-103,81,216,35,119,191,-1
5323,219,125,92,231,-63,285,17,-34,82,24,279,135,159,128,-81,235,235,9,69,59
2578,157,94,84,-2,257,253,-168,310,122,69,-11,-9,-82,-8,207,118,34,245,75,196
7695,178,29,75,231,23,316,-61,28,89,-82,-42,-13,137,342,-68,56,182,93,145,211


### preprocess data

In [13]:
# set up standard scalar preprocessing
scaler = StandardScaler()

transformer_stdscaler = scaler.fit(X_train) # Q which should we use? train? test? oot? all?

# transform data
X_train_processed = transformer_stdscaler.transform(X_train)
X_test_processed = transformer_stdscaler.transform(X_test)
X_oot_processed = transformer_stdscaler.transform(X_oot)

print('X_train_processed', X_train_processed.shape[0])
print('X_test_processed', X_test_processed.shape[0])
print('X_oot_processed', X_oot_processed.shape[0])

pd.DataFrame(X_train_processed)

X_train_processed 4766
X_test_processed 1192
X_oot_processed 2518


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1.255214,1.388927,1.643464,0.724691,1.504530,-0.692663,1.674089,-1.047813,-0.333712,-0.138527,1.252147,0.558506,-0.105877,-0.654908,0.215012,0.827151,0.925081,-0.800581,-0.532266,0.596357
1,0.771374,-1.347159,-0.283924,-1.368465,0.112051,0.325777,-1.043418,0.969164,-0.243111,-1.267360,-0.626909,-0.073810,-1.264164,1.070990,0.134461,0.522867,-0.297774,1.415282,-0.135183,0.206710
2,0.307283,-2.209077,0.365094,-0.667408,0.374402,0.245899,1.257537,-1.986406,0.874302,0.430797,1.532007,-0.214325,-1.125566,-0.455382,-2.251884,0.130243,-1.254363,-0.908912,-0.609745,0.016638
3,-0.008693,0.065984,0.463430,0.003604,0.667025,0.076159,1.574910,-0.688352,-1.783329,-0.374110,-1.076684,-0.846640,1.349408,0.941298,0.990323,0.287293,0.254483,-0.091505,-1.336116,-1.513439
4,-0.275299,1.038146,-1.326288,-0.116577,-0.321837,0.056190,-0.349164,-3.194595,-0.766584,-1.277176,-0.267090,-0.786420,-2.719449,-1.891968,-1.355746,0.562129,-0.564041,0.263033,1.201340,0.539336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4761,1.679808,0.667322,0.876442,0.955038,-2.400465,-0.672694,1.346798,0.080496,0.632699,-0.639140,1.532007,-0.294619,0.725715,-2.001707,-0.207884,1.141250,-0.652797,0.174399,0.891422,-0.943224
4762,1.146597,0.216318,-0.126587,1.275522,-1.704225,1.843452,-0.924403,-1.447214,-0.313578,-0.933618,1.791876,0.357771,0.567316,0.302816,-1.839057,1.327746,1.319551,-0.908912,-0.290142,-0.373009
4763,0.534392,-0.094373,-0.205256,-1.057997,1.524711,1.523942,-2.759216,1.987638,0.089093,-0.491901,-1.106669,-1.087522,-1.818558,-1.053959,1.060805,0.179321,-0.662659,1.415282,-0.232032,0.928983
4764,0.741752,-0.745822,-0.293758,1.275522,-0.836449,2.152978,-1.698000,-0.828142,-0.243111,-1.974108,-1.416513,-1.127669,0.349519,2.437741,-1.708160,-0.429246,0.796879,-0.081656,0.445914,1.071536


### Train Model

In [14]:
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression
lr_clf = LogisticRegression(random_state=88, max_iter=1000)
lr_clf.fit(X_train_processed, y_train)

# Evaluate on train / test / oot
lr_train_auc = roc_auc_score(y_train, lr_clf.predict_proba(X_train_processed)[:, 1])
lr_test_auc  = roc_auc_score(y_test,  lr_clf.predict_proba(X_test_processed)[:, 1])
lr_oot_auc   = roc_auc_score(y_oot,   lr_clf.predict_proba(X_oot_processed)[:, 1])

print("=== Logistic Regression ===")
print(f"Train AUC : {lr_train_auc:.4f}  |  GINI: {round(2*lr_train_auc-1, 3)}")
print(f"Test  AUC : {lr_test_auc:.4f}  |  GINI: {round(2*lr_test_auc-1, 3)}")
print(f"OOT   AUC : {lr_oot_auc:.4f}  |  GINI: {round(2*lr_oot_auc-1, 3)}")


=== Logistic Regression ===
Train AUC : 0.6500  |  GINI: 0.3
Test  AUC : 0.6490  |  GINI: 0.298
OOT   AUC : 0.6253  |  GINI: 0.251


In [15]:
# --- save model artefact to model bank ---
model_name = model_name = "lr_model_2024_12_01.pkl"
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

model_artefact = {
    "model": lr_clf,
    "preprocessing_transformers": {
        "stdscaler": transformer_stdscaler
    }
}

model_artefact_filepath = model_bank_directory + model_name
with open(model_artefact_filepath, 'wb') as f:
    pickle.dump(model_artefact, f)

print("Model saved to:", model_artefact_filepath)


Model saved to: model_bank/lr_model_2024_12_01.pkl
